In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, viewer
import numpy as np
import sim_utils, param_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow
from Stretch2Relax import extra_utils

# Load Meshes and compute Tutte embedding

In [ ]:
m_rest = mesh.Mesh('../Stretch2Relax/ToysMesh/Hilbert_opt_2d.obj')
m_defo = mesh.Mesh('../Stretch2Relax/ToysMesh/Hilbert_init_2d.obj')
v = mesh_energy.NodalVars(m_rest, 2)
v.setVars(m_defo.vertices().ravel())

In [ ]:
nf = newton_flow.symmetric_dirichlet(m_rest, v)
# nf = newton_flow.linear_elastic(m_rest, v)

In [ ]:
import py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
# TODO: add epsilon * low rank.
FIX_VARS = False
if FIX_VARS:
    fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    prob.setFixedVars(fv)
else:
    # prob.hessianShift = 1e-5
    # nf.elementHessianShift = 1e-6
    prob.hessianShift = 1e-10
    prob.useRelativeHessianShift = True

In [ ]:
import newton_flow_utils

In [ ]:
eval_traj = {'logspiral': newton_flow_utils.eval_trajectory_logspiral,
             'taylor': newton_flow_utils.eval_trajectory_taylor,
             'componentwise_pade': newton_flow_utils.eval_trajectory_componentwise_pade,
             'vector_pade': newton_flow_utils.eval_trajectory_vector_pade}

In [ ]:
extrapolation_dist = 4
constant_speed = True
always_project = True
trajectory_type = 'vector_pade' # 'componentwise_pade' # 'taylor' # 'logspiral'

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

## Possion-based Extrapolate

In [ ]:
m_rest_3 = mesh.Mesh('../Stretch2Relax/ToysMesh/Hilbert_opt_2d.obj', embeddingDimension=3)
param_poisson, prob_poisson = extra_utils.getParamProb(m_rest_3, m_defo.vertices(), FIX_VARS=FIX_VARS)

opt2 = prob_poisson.optimizer()
opt2.options.hessianProjectionController.startWithProjectionActive = False
opt2.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt2.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt2.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
prob_poisson.hessianShift = 1e-10
prob_poisson.useRelativeHessianShift = True

# opt2.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
# opt2.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
# opt2.options.hessianProjectionController.startWithProjectionActive = False

In [ ]:
Linv = extra_utils.getLaplacianFactorizer(m_rest_3, fixedVars=[0])

### For NewtonFlow Baseline comparisons

In [ ]:
def eval_linear_extrapolate(x_0, coeffs, alphas):
    return np.array([(x_0 + coeffs[0] * a).reshape(-1, 2) for a in alphas])

def eval_poisson_based_extrapolate(x_0, coeffs, alphas):    
    prob_poisson.setVars(x_0)
    uvs_extra = []
    uv_centrial = x_0.reshape(-1,2).mean(axis=0)
    for a in alphas:
        uv_new = extra_utils.paramNewtonstepExtrapolation(m_rest_3, coeffs[0], param_poisson, a, Linv, method = 'Eulerian', fixedVind=None, fixedUV=m_defo.vertices()[90])
        uv_new_centrial = uv_new.mean(axis=0)
        uv_new += uv_centrial - uv_new_centrial
        uvs_extra.append(uv_new)
    return np.array(uvs_extra)

## Newton Iterations first

In [ ]:
opt2.options.niter = 50

In [ ]:
opt2.optimize()

In [ ]:
prob.setVars(prob_poisson.getVars())

In [ ]:
visualization.plot_mesh(prob_poisson.getVars().reshape(-1,2), m_rest.elements())

## Video Compare

In [ ]:
import visualization
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
import newton_flow_utils as nfu
fv = nfu.ground_truth_flow(opt, 0.02, grad_tol=1e-5)

In [ ]:
extrapolation_dist = 20
constant_speed = True
num_frames = min(500, len(fv))

baseline_methods = [(1, nfu.eval_trajectory_taylor, 'Newton'),
           (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
           (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor')]

baseline_methods = []

ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(1, eval_poisson_based_extrapolate, 'Poisson')])


In [ ]:
ff(0)

In [ ]:
ff(100)

In [ ]:
ff(200)

In [ ]:
brek

In [ ]:
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(2, nfu.eval_trajectory_logspiral, 'Deg 2 Spiral')])
visualization.writeVideo('spiral_deg_2_compare.mp4', num_frames, ff)

ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(3, nfu.eval_trajectory_logspiral, 'Deg 3 Spiral')])
visualization.writeVideo('spiral_deg_3_compare.mp4', num_frames, ff)

extrapolation_dist = 5
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(14, nfu.eval_trajectory_vector_pade, 'Deg 14 Pade')],
                         truncate=True)
visualization.writeVideo('pade_compare.mp4', num_frames, ff)

In [ ]:
# Simple automated method:
# Try search over degree (up to 4) at alpha = 1; backtrack if necessary.
# Test verson with projected Hessian.

# TODO
- Postprocess Newton step to remove rigid motion (does this make the steps more coherent?)
- Try KKT formulation for pinning rigid motion
- See relative performance of symmetric indefinite factorization within Accelerate
- Consider sparse + epsilon * low rank factorization idea for pinning rigid motion (omitting epsilon^2 fully dense term)

# Finite Difference Validation

In [ ]:
proj = False

In [ ]:
opt.update_factorizations()
d = opt.newton_step()
d_coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, proj)

In [ ]:
import benchmark
prob.disableCaching = True
benchmark.reset()
x = prob.getVars()
eps = 0.001
prob.setVars(x + eps * d)
d_plus = opt.newton_step()
d_coeffs_plus = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, proj)

prob.setVars(x - eps * d)
d_minus = opt.newton_step()
d_coeffs_minus = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, proj)
d_prime_ad = (d_plus - d_minus) / (2 * eps)
# Note: this finite difference approximation is wrong! we must incorporate the effect of `d_prime_ad` when differencing!
d_pprime_ad = (d_plus + d_minus - 2 * d) / (eps * eps) 

prob.setVars(x + eps * d + 0.5 * (eps * eps) * d_prime_ad)
d_pp = opt.newton_step()
prob.setVars(x - eps * d + 0.5 * (eps * eps) * d_prime_ad)
d_mm = opt.newton_step()
d_pprime_ad2 = (d_pp + d_mm - 2 * d) / (eps * eps)
prob.setVars(x)
# benchmark.report()

In [ ]:
import math
i = 1
((d_coeffs_plus[i][:5] - d_coeffs_minus[i][:5]) / (2 * eps)) / (d_coeffs[i + 1][:5] * math.factorial(i + 2) / math.factorial(i + 1))

In [ ]:
opt.update_factorizations()
speed_factor = 2
d_coeffs = [x_i * speed_factor**(i + 1) for i, x_i in enumerate(nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, 16, proj))]

In [ ]:
import benchmark
prob.disableCaching = True
benchmark.reset()
eps = 0.0001
prob.setVars(x + eps * d_coeffs[0])
opt.update_factorizations()
d_coeffs_plus = [x_i * speed_factor**(i + 1) for i, x_i in enumerate(nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, 16, proj))]

prob.setVars(x - eps * d_coeffs[0])
opt.update_factorizations()
d_coeffs_minus = [x_i * speed_factor**(i + 1) for i, x_i in enumerate(nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, 16, proj))]
prob.setVars(x)

In [ ]:
import math
i = 11
((d_coeffs_plus[i][:5] - d_coeffs_minus[i][:5]) / (2 * eps)) / (d_coeffs[i + 1][:5] * math.factorial(i + 2) / math.factorial(i + 1))